# Chapter 3 — Working with LLMs

## Setup Instructions

To ensure you have the required dependencies to run this notebook, you'll need to have our `llm-agents-from-scratch` framework installed on the running Jupyter kernel. To do this, you can launch this notebook with the following command while within the project's root directory:

```sh
uv run --with jupyter jupyter lab
```

Alternatively, if you just want to use the published version of `llm-agents-from-scratch` without local development, you can install it from PyPi by uncommenting the cell below.

<!-- provider-note:start -->
## Setting the backbone LLM of your agent

These notebooks run on **Ollama by default**, the setup the book teaches.
If you do nothing, nothing changes: `make_llm()` starts a local Ollama
service when one isn't already running.

To use OpenAI or Anthropic instead:

1. Install the extra: `uv sync --extra openai` or `uv sync --extra anthropic`.
2. Export `OPENAI_API_KEY` or `ANTHROPIC_API_KEY` before launching Jupyter.
3. Set `LLM_PROVIDER=openai` (or `anthropic`), or pass `provider="openai"`
   to `make_llm()`. A key on its own never switches providers, so one
   exported for unrelated work cannot reroute you off the Ollama path.

The switch applies wherever a notebook builds its LLM with `make_llm()`.
A notebook that constructs `OllamaLLM` directly stays on Ollama regardless
of these settings.

If you opted in but forgot to export the key, you will be prompted for it.
That is the safe path on hosted kernels, and it keeps the key out of the
saved notebook. Setting `OLLAMA_API_KEY` alone routes Ollama to Ollama
Cloud.

**Caveat:** the examples are tuned for qwen3. Output on gpt-5 or Claude
will differ from what is printed in the book, and prompt-sensitive
examples (the ch09 evaluator pattern, ch08 supervised trajectories) may
behave noticeably differently.
<!-- provider-note:end -->

In [1]:
# Uncomment the line below to install `llm-agents-from-scratch` from PyPi
# !pip install llm-agents-from-scratch

## Running an Ollama service

To execute the code provided in this section, you’ll need to have Ollama installed on your local machine and have its LLM hosting service running. To download Ollama, follow the instructions found on this page: https://ollama.com/download. After downloading and installing Ollama, you can start a service by opening a terminal and running the command `ollama serve`.

In [2]:
from llm_agents_from_scratch.notebook_utils import ollama_settings

settings = ollama_settings()

✓ Using Ollama Cloud (kimi-k2.7-code:cloud)


## Examples

The code in the book runs its coroutines with `asyncio.run()`. A Jupyter kernel already has an event loop running, so this notebook awaits the same coroutines directly at the top level of a cell instead.

### Example 1: Joke model for `structured_output()`

In [3]:
from typing import Literal
from pydantic import BaseModel

class Joke(BaseModel):
    """A structured output model for Jokes."""

    subject: Literal["math", "physics", "biology"]
    joke: str

### Example 2: Instantiating an `OllamaLLM`

Note: Examples 2 through 5 construct `OllamaLLM` directly, because the constructor is the point of these examples. `settings` comes from the setup cell above and resolves the model and host the same way `make_llm()` does in every other notebook: Ollama Cloud when `OLLAMA_API_KEY` is set, otherwise a local Ollama service running `qwen3:14b`. `LLM_PROVIDER` has no effect here, since these examples are about `OllamaLLM`.

In [4]:
from llm_agents_from_scratch.llms.ollama import OllamaLLM

llm = OllamaLLM(**settings)

### Example 3: Using `complete()` to have an `OllamaLLM` tell a joke

In [5]:
from llm_agents_from_scratch.llms.ollama import OllamaLLM

llm = OllamaLLM(**settings)
response = await llm.complete("Tell me a joke.")
print(response)

response='Why don’t scientists trust atoms?\n\nBecause they make up everything.' prompt='Tell me a joke.'


### Example 4: Using `structured_output()` to have an `OllamaLLM` tell a (structured) joke

In [6]:
llm = OllamaLLM(**settings)
prompt = "Tell me a joke."
joke = await llm.structured_output(prompt=prompt, mdl=Joke)
print(joke.__class__.__name__)
print(joke)

Joke
subject='physics' joke='Why did the photon refuse to check a suitcase at the airport? Because it was traveling light.'


### Example 5: Hailstone tool call with `OllamaLLM`

In [7]:
def hailstone_step_func(x: int) -> int:
    """Performs a single step of the Hailstone sequence."""
    if x % 2 == 0:
        return x // 2
    return 3 * x + 1

#### Eliciting a tool call request

In [8]:
from llm_agents_from_scratch.llms.ollama import OllamaLLM
from llm_agents_from_scratch.data_structures.llm import ChatMessage
from llm_agents_from_scratch.tools import SimpleFunctionTool

hailstone_tool = SimpleFunctionTool(hailstone_step_func)
llm = OllamaLLM(**settings)

user_input = (
    "What is the result of taking the next step of the "
    "Hailstone sequence on the number 3?\n\n"
    "Be very succinct in your response."
)
user_msg, response_msg = await llm.chat(
    user_input,
    tools=[hailstone_tool],
)
print(response_msg.tool_calls)

[ToolCall(id_='7ddc4596-68a8-4a2c-b9a4-d6f715829239', tool_name='hailstone_step_func', arguments={'x': 3})]


#### Executing the tool

In [9]:
tool_call = response_msg.tool_calls[0]
tool_call_result = hailstone_tool(tool_call) # a ToolCallResult
print(tool_call_result)

tool_call_id='7ddc4596-68a8-4a2c-b9a4-d6f715829239' content='10' error=False


#### Sending tool results back to LLM

In [10]:
tools_msg, final_response = await llm.continue_chat_with_tool_results(
    tool_call_results=[tool_call_result],
    chat_history=[user_msg, response_msg],
)
print(final_response)

role=<ChatRole.ASSISTANT: 'assistant'> content='10' tool_calls=None


## Optional providers: `OpenAILLM` and `AnthropicLLM`

`OllamaLLM` is the LLM the book builds on, but it is not the only one. The framework ships two optional providers that implement the same `BaseLLM` interface you used above, so `complete()`, `structured_output()`, `chat()` and `continue_chat_with_tool_results()` work the same way on each of them:

| Class | Extra | Environment variable |
|---|---|---|
| `OpenAILLM` | `uv sync --extra openai` | `OPENAI_API_KEY` |
| `AnthropicLLM` | `uv sync --extra anthropic` | `ANTHROPIC_API_KEY` |

Constructing one looks just like Example 2:

```python
from llm_agents_from_scratch.llms.openai import OpenAILLM
from llm_agents_from_scratch.llms.anthropic import AnthropicLLM

llm = OpenAILLM(model="gpt-5")
llm = AnthropicLLM(model="claude-sonnet-5")
```

From Chapter 4 onward the notebooks stop naming a class and call `make_llm()` instead, which builds whichever of these three the `LLM_PROVIDER` setting asks for (see the note at the top of this notebook). Anything that accepts an `LLM` in later chapters, starting with `LLMAgent`, accepts any of them.